In [ ]:
# Load artifacts from Preprocessing_&_EDA.ipynb

import json
import random
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler

import lightgbm as lgb
import xgboost as xgb

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    BatchNormalization, Bidirectional, Conv1D, Dense, Dropout,
    GlobalAveragePooling1D, GRU, Input, LSTM,
    LayerNormalization, MultiHeadAttention,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
logging.getLogger("lightgbm").setLevel(logging.ERROR)

ARTIFACT_CANDIDATES = [
    Path("/content/outputs/preprocessing_artifacts"),
    Path("/content/drive/MyDrive/outputs/preprocessing_artifacts"),
    Path("/content/drive/MyDrive/preprocessing_artifacts"),
    Path("outputs/preprocessing_artifacts"),
    Path("preprocessing_artifacts"),
]

ARTIFACT_DIR = next(
    (p for p in ARTIFACT_CANDIDATES if (p / "engineered_df.csv").exists()),
    ARTIFACT_CANDIDATES[0],
)

ENGINEERED_DATA_PATH = ARTIFACT_DIR / "engineered_df.csv"
PROFILE_PATH = ARTIFACT_DIR / "profile.json"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

if not ENGINEERED_DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find engineered_df.csv. Run Preprocessing_&_EDA.ipynb first "
        "and make sure its preprocessing_artifacts folder is available here."
    )

engineered_df = pd.read_csv(ENGINEERED_DATA_PATH)

with open(PROFILE_PATH, "r", encoding="utf-8") as f:
    profile = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

FINAL_TEST_START = int(metadata.get("final_test_start", 2020))
RANDOM_STATE = int(metadata.get("random_state", 42))
WINDOW = int(metadata.get("window", 3))
DL_N_RUNS = int(metadata.get("dl_n_runs", 10))
MAX_EPOCHS = int(metadata.get("max_epochs", 150))
DL_VERBOSE = int(metadata.get("dl_verbose", 0))
CLIMATE_COLS = [c for c in metadata.get("climate_cols", []) if c in engineered_df.columns]

OUTPUT_DIR = Path("/content/outputs") if Path("/content").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

engineered_df["YEAR"] = engineered_df["YEAR"].astype(int)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except Exception:
    pass

print(f"Loaded artifacts from: {ARTIFACT_DIR}")
print(f"engineered_df shape : {engineered_df.shape}")
print(f"Year range          : {engineered_df['YEAR'].min()}-{engineered_df['YEAR'].max()}")
print(f"Climate columns     : {CLIMATE_COLS}")

In [ ]:
#Helper functions

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

def finite_or_none(values: np.ndarray) -> np.ndarray | None:
    arr = np.asarray(values, dtype=float)
    return arr if np.all(np.isfinite(arr)) else None

def slugify(name: str) -> str:
    return name.lower().replace(" ", "_").replace("-", "_")

def build_feature_columns(df: pd.DataFrame) -> list:
    drop_cols = {"PRODUCTION", "YIELD"}
    return [col for col in df.columns if col not in drop_cols]

def build_preprocessor(
    feature_df: pd.DataFrame,
    categorical_cols: list | None = None,
) -> tuple:
    if categorical_cols is None:
        categorical_cols = [col for col in ["DISTRICT", "CROP"] if col in feature_df.columns]

    numeric_cols = [col for col in feature_df.columns if col not in categorical_cols]

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("power", PowerTransformer(method="yeo-johnson", standardize=True)),
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])

    return preprocessor, numeric_cols, categorical_cols

def add_train_only_anomalies(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    anomaly_cols: Iterable[str],
    group_cols: list | None = None,
) -> tuple:
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    anomaly_cols = [col for col in anomaly_cols if col in train_df.columns]

    if group_cols is None:
        group_cols = ["DISTRICT", "CROP"]

    if not anomaly_cols:
        return train_df, valid_df

    group_means = train_df.groupby(group_cols)[anomaly_cols].mean().reset_index()
    global_means = train_df[anomaly_cols].mean()

    train_df = train_df.merge(group_means, on=group_cols, how="left", suffixes=("", "_group_mean"))
    valid_df = valid_df.merge(group_means, on=group_cols, how="left", suffixes=("", "_group_mean"))

    for col in anomaly_cols:
        mean_col = f"{col}_group_mean"
        train_ref = train_df[mean_col].fillna(global_means[col])
        valid_ref = valid_df[mean_col].fillna(global_means[col])
        train_df[f"{col}_anomaly"] = train_df[col] - train_ref
        valid_df[f"{col}_anomaly"] = valid_df[col] - valid_ref

    drop_cols = [f"{col}_group_mean" for col in anomaly_cols]
    train_df = train_df.drop(columns=drop_cols, errors="ignore")
    valid_df = valid_df.drop(columns=drop_cols, errors="ignore")

    return train_df, valid_df

def build_tabular_arrays(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    categorical_cols: list | None = None,
) -> tuple:
    feature_cols = build_feature_columns(train_df)
    preprocessor, _, _ = build_preprocessor(
        train_df[feature_cols],
        categorical_cols=categorical_cols,
    )

    X_train = preprocessor.fit_transform(train_df[feature_cols])
    X_valid = preprocessor.transform(valid_df[feature_cols])
    y_train = train_df["YIELD"].to_numpy()
    y_valid = valid_df["YIELD"].to_numpy()

    return X_train, X_valid, y_train, y_valid, feature_cols

---
## Model Definitions

### Traditional ML Models

In [ ]:
def make_models(preprocessor: ColumnTransformer) -> dict:
    """
    Instantiate all sklearn models wrapped in TransformedTargetRegressor.
    Target is Yeo-Johnson transformed before fitting and back-transformed before prediction.

    Models: RF, GB, XGB, LGB
    """
    return {

        "RF": TransformedTargetRegressor(
            regressor=Pipeline(steps=[
                ("prep",  clone(preprocessor)),
                ("model", RandomForestRegressor(
                    n_estimators=500, min_samples_leaf=2,
                    random_state=RANDOM_STATE, n_jobs=1,
                )),
            ]),
            transformer=PowerTransformer(method="yeo-johnson", standardize=False),
        ),
        "GB": TransformedTargetRegressor(
            regressor=Pipeline(steps=[
                ("prep",  clone(preprocessor)),
                ("model", GradientBoostingRegressor(
                    learning_rate=0.05, max_depth=6,
                    min_samples_leaf=10, random_state=RANDOM_STATE,
                )),
            ]),
            transformer=PowerTransformer(method="yeo-johnson", standardize=False),
        ),
        "XGB": TransformedTargetRegressor(
            regressor=Pipeline(steps=[
                ("prep",  clone(preprocessor)),
                ("model", xgb.XGBRegressor(
                    n_estimators=300, learning_rate=0.05, max_depth=6,
                    subsample=0.9, colsample_bytree=0.9,
                    random_state=RANDOM_STATE, n_jobs=1,
                    verbosity=0, objective="reg:squarederror",
                )),
            ]),
            transformer=PowerTransformer(method="yeo-johnson", standardize=False),
        ),
        "LGB": TransformedTargetRegressor(
            regressor=Pipeline(steps=[
                ("prep",  clone(preprocessor)),
                ("model", lgb.LGBMRegressor(
                    n_estimators=300, learning_rate=0.05, max_depth=6,
                    num_leaves=31, random_state=RANDOM_STATE,
                    n_jobs=1, verbose=-1,
                )),
            ]),
            transformer=PowerTransformer(method="yeo-johnson", standardize=False),
        ),
    }


### Deep Learning — Sequence Models (LSTM, GRU, CNN, Hybrids)

> **Architecture notes:**  
> - `BatchNormalization` added to CNN-based models for stable convergence.  
> - `recurrent_dropout` added to LSTM/GRU to regularise temporal connections.  
> - `ReduceLROnPlateau` added alongside `EarlyStopping` for smoother convergence.  
> - All models averaged over `DL_N_RUNS` independent seeds.


In [ ]:
def make_sequence_arrays(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    window: int = WINDOW,
    group_cols: list | None = None,
    categorical_cols: list | None = None,
) -> tuple:
    """
    Build sliding-window 3-D arrays (samples x timesteps x features) for sequence models.

    Fixes applied:
    - Numeric scaling is fit on training rows only, then applied to validation rows.
    - Each sample window ends at the prediction year, so sequence models can use the
      same current-year climate features available to the tabular models.
    """
    if group_cols is None:
        group_cols = ["DISTRICT", "CROP"]
    if categorical_cols is None:
        categorical_cols = [col for col in ["DISTRICT", "CROP"] if col in train_df.columns]

    seq_feature_cols = [
        col for col in build_feature_columns(train_df)
        if col not in categorical_cols + ["YEAR"]
    ]

    train_block = train_df.copy().sort_values(group_cols + ["YEAR"]).reset_index(drop=True)
    valid_block = valid_df.copy().sort_values(group_cols + ["YEAR"]).reset_index(drop=True)
    train_block["_split"] = "train"
    valid_block["_split"] = "valid"

    x_scaler = StandardScaler()
    train_numeric = pd.DataFrame(
        x_scaler.fit_transform(train_block[seq_feature_cols]),
        columns=seq_feature_cols,
        index=train_block.index,
    )
    valid_numeric = pd.DataFrame(
        x_scaler.transform(valid_block[seq_feature_cols]),
        columns=seq_feature_cols,
        index=valid_block.index,
    )

    if categorical_cols:
        train_dummies = pd.get_dummies(train_block[categorical_cols], columns=categorical_cols)
        valid_dummies = pd.get_dummies(valid_block[categorical_cols], columns=categorical_cols)
    else:
        train_dummies = pd.DataFrame(index=train_block.index)
        valid_dummies = pd.DataFrame(index=valid_block.index)

    dummy_cols = sorted(set(train_dummies.columns).union(valid_dummies.columns))
    for frame in [train_dummies, valid_dummies]:
        for col in dummy_cols:
            if col not in frame.columns:
                frame[col] = 0
    train_dummies = train_dummies[dummy_cols]
    valid_dummies = valid_dummies[dummy_cols]

    train_seq_matrix = pd.concat(
        [
            train_block[group_cols + ["YEAR", "_split", "YIELD"]].reset_index(drop=True),
            train_numeric.reset_index(drop=True),
            train_dummies.reset_index(drop=True),
        ],
        axis=1,
    )
    valid_seq_matrix = pd.concat(
        [
            valid_block[group_cols + ["YEAR", "_split", "YIELD"]].reset_index(drop=True),
            valid_numeric.reset_index(drop=True),
            valid_dummies.reset_index(drop=True),
        ],
        axis=1,
    )
    seq_matrix = pd.concat([train_seq_matrix, valid_seq_matrix], ignore_index=True)

    X_train_s, y_train_s, X_valid_s, y_valid_s = [], [], [], []
    for _, grp in seq_matrix.groupby(group_cols):
        grp = grp.sort_values("YEAR").reset_index(drop=True)
        feature_block = grp.drop(columns=group_cols + ["YEAR", "_split", "YIELD"]).to_numpy(dtype=np.float32)
        target = grp["YIELD"].to_numpy(dtype=np.float32)
        split = grp["_split"].to_numpy()

        if len(grp) < window:
            continue

        for idx in range(window - 1, len(grp)):
            x_s = feature_block[idx - window + 1 : idx + 1]
            y_ = target[idx]
            if split[idx] == "train":
                X_train_s.append(x_s)
                y_train_s.append(y_)
            else:
                X_valid_s.append(x_s)
                y_valid_s.append(y_)

    return (
        np.array(X_train_s, dtype=np.float32),
        np.array(y_train_s, dtype=np.float32),
        np.array(X_valid_s, dtype=np.float32),
        np.array(y_valid_s, dtype=np.float32),
    )


In [ ]:
# Sequence model architectures (right-sized for small panels)

def build_lstm_model(input_shape: tuple) -> Sequential:
    """Stacked LSTM with recurrent_dropout: 32 → 16 → Dense(16) → 1."""
    model = Sequential([
        Input(shape=input_shape),
        LSTM(32, return_sequences=True, recurrent_dropout=0.1),
        Dropout(0.2),
        LSTM(16, recurrent_dropout=0.1),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(0.001), loss="mse")
    return model

def build_gru_model(input_shape: tuple) -> Sequential:
    """Stacked GRU with recurrent_dropout: 32 → 16 → Dense(16) → 1."""
    model = Sequential([
        Input(shape=input_shape),
        GRU(32, return_sequences=True, recurrent_dropout=0.1),
        Dropout(0.2),
        GRU(16, recurrent_dropout=0.1),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(0.001), loss="mse")
    return model

def build_cnn1d_model(input_shape: tuple) -> Sequential:
    """1-D CNN with BatchNormalization and global average pooling."""
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, kernel_size=2, activation="relu", padding="same"),
        BatchNormalization(),
        Dropout(0.2),
        Conv1D(16, kernel_size=2, activation="relu", padding="same"),
        BatchNormalization(),
        GlobalAveragePooling1D(),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(0.001), loss="mse")
    return model

def build_rcnn_model(input_shape: tuple) -> Sequential:
    """Recurrent-CNN: Conv1D feature extraction → LSTM sequence modelling."""
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, kernel_size=2, activation="relu", padding="same"),
        BatchNormalization(),
        Dropout(0.15),
        LSTM(16, recurrent_dropout=0.1),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(0.001), loss="mse")
    return model

def build_cnn_bilstm_model(input_shape: tuple) -> Sequential:
    """CNN + Bidirectional LSTM hybrid."""
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, kernel_size=2, activation="relu", padding="same"),
        BatchNormalization(),
        Dropout(0.15),
        Bidirectional(LSTM(16, recurrent_dropout=0.1)),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(0.001), loss="mse")
    return model

def build_cnn_transformer_model(input_shape: tuple) -> Model:
    """CNN + Transformer encoder with multi-head self-attention.
    clipnorm=1.0 prevents gradient explosion in the residual attention block.
    """
    inputs  = Input(shape=input_shape)
    x       = Conv1D(32, kernel_size=2, activation="relu", padding="same")(inputs)
    x       = BatchNormalization()(x)
    x       = Dropout(0.15)(x)
    attn    = MultiHeadAttention(num_heads=2, key_dim=16, dropout=0.1)(x, x)
    x       = LayerNormalization(epsilon=1e-6)(x + attn)
    ff      = Dense(32, activation="relu")(x)
    ff      = Dense(32)(ff)
    x       = LayerNormalization(epsilon=1e-6)(x + ff)
    x       = GlobalAveragePooling1D()(x)
    x       = Dropout(0.2)(x)
    x       = Dense(16, activation="relu")(x)
    outputs = Dense(1)(x)
    model   = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001, clipnorm=1.0), loss="mse")
    return model

SEQUENCE_MODELS = {
    "LSTM"           : build_lstm_model,
    "GRU"            : build_gru_model,
    "1DCNN"          : build_cnn1d_model,
    "RCNN"           : build_rcnn_model,
    "CNN-BiLSTM"     : build_cnn_bilstm_model,
    "CNN-Transformer": build_cnn_transformer_model,
}


In [ ]:
def fit_seq_runs(
    builder,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    y_valid: np.ndarray,
    n_runs: int = DL_N_RUNS,
    model_label: str = "SEQ",
    stage_label: str = "",
) -> tuple:
    """
    Train a sequence model n_runs times and return per-run metrics plus mean prediction.
    Training stops via EarlyStopping; MAX_EPOCHS is only a safety cap.
    """
    y_scaler = StandardScaler()
    y_train_sc = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-5, verbose=0),
    ]

    pred_runs = []
    run_rows = []
    for run in range(n_runs):
        prefix = f"[{stage_label}] " if stage_label else ""
        print(f"{prefix}{model_label} run {run+1}/{n_runs} started")
        tf.random.set_seed(RANDOM_STATE + run)
        model = builder((X_train.shape[1], X_train.shape[2]))
        history = model.fit(
            X_train,
            y_train_sc,
            validation_split=0.15,
            epochs=MAX_EPOCHS,
            batch_size=16,
            callbacks=callbacks,
            verbose=DL_VERBOSE,
        )
        preds_sc = model.predict(X_valid, verbose=0).ravel()
        preds = y_scaler.inverse_transform(preds_sc.reshape(-1, 1)).ravel()
        epochs_used = len(history.history.get("loss", []))
        if np.all(np.isfinite(preds)):
            metrics = metrics_dict(y_valid, preds)
            pred_runs.append(preds)
            run_rows.append({"run": run + 1, "epochs_used": epochs_used, **metrics})
            print(
                f"{prefix}{model_label} run {run+1}/{n_runs} finished: "
                f"epochs={epochs_used}  RMSE={metrics['rmse']:.4f}  "
                f"MAE={metrics['mae']:.4f}  R2={metrics['r2']:.4f}"
            )
        else:
            print(f"{prefix}{model_label} run {run+1}/{n_runs} finished: epochs={epochs_used}  non-finite predictions, skipped")
        tf.keras.backend.clear_session()

    run_df = pd.DataFrame(run_rows)
    mean_preds = np.mean(pred_runs, axis=0) if pred_runs else np.full(len(X_valid), np.nan)
    return run_df, mean_preds


---
## Temporal Cross-Validation

> **Rolling-origin strategy:** training window grows year by year; validation is always 2 years
> ahead of the training cut-off. Only years strictly before `FINAL_TEST_START` are used —
> the holdout test block is never touched here.


In [ ]:
@dataclass
class FoldResult:
    """Stores metrics for a single model on a single CV fold."""
    model              : str
    fold_train_end_year: int
    validation_years   : str
    rmse               : float
    mae                : float
    r2                 : float

def temporal_folds(train_df: pd.DataFrame) -> list:
    """
    Define rolling-origin folds.
    Fold end years: 2013, 2015, 2017
    Each fold validates on the 2 years immediately following the cut-off.
    Folds whose validation years overlap the final test block are discarded.
    """
    fold_end_years = [2013, 2015, 2017]
    folds = []
    for end_year in fold_end_years:
        val_years  = np.array([end_year + 1, end_year + 2])
        if val_years.max() >= FINAL_TEST_START:
            continue
        train_mask = train_df["YEAR"] <= end_year
        val_mask   = train_df["YEAR"].isin(val_years)
        if train_mask.sum() == 0 or val_mask.sum() == 0:
            continue
        folds.append((end_year, train_mask.to_numpy(), val_mask.to_numpy()))
    return folds


In [ ]:
def run_cross_validation(
    train_df: pd.DataFrame,
    feature_cols: list,
    group_cols: list | None = None,
    categorical_cols: list | None = None,
) -> tuple:
    """
    Run all models on all temporal folds and collect metrics.

    Returns
    -------
    cv_df      : one averaged row per (model, fold)
    dl_runs_df : one row per DL run per fold
    """
    if group_cols is None:
        group_cols = ["DISTRICT", "CROP"]

    folds = temporal_folds(train_df)
    fold_rows = []
    dl_run_rows = []

    for fold_end_year, train_mask, val_mask in folds:
        fold_train = train_df.loc[train_mask].copy()
        fold_val = train_df.loc[val_mask].copy()
        stage_label = f"CV {fold_end_year+1}-{fold_end_year+2}"
        print(f"\n--- {stage_label} ---")

        fold_train, fold_val = add_train_only_anomalies(
            fold_train,
            fold_val,
            anomaly_cols=CLIMATE_COLS,
            group_cols=group_cols,
        )

        fold_feature_cols = build_feature_columns(fold_train)
        preprocessor, _, _ = build_preprocessor(
            fold_train[fold_feature_cols], categorical_cols=categorical_cols
        )
        models = make_models(preprocessor)

        fitted_models = {}
        for model_name, model in models.items():
            model.fit(fold_train[fold_feature_cols], fold_train["YIELD"])
            fitted_models[model_name] = model
            preds = finite_or_none(model.predict(fold_val[fold_feature_cols]))
            if preds is None:
                continue
            m = metrics_dict(fold_val["YIELD"].to_numpy(), preds)
            fold_rows.append(FoldResult(
                model=model_name,
                fold_train_end_year=fold_end_year,
                validation_years=f"{fold_end_year+1}-{fold_end_year+2}",
                **m,
            ))

        if "RF" in fitted_models and "XGB" in fitted_models:
            preds = finite_or_none((
                fitted_models["RF"].predict(fold_val[fold_feature_cols])
                + fitted_models["XGB"].predict(fold_val[fold_feature_cols])
            ) / 2.0)
            if preds is not None:
                m = metrics_dict(fold_val["YIELD"].to_numpy(), preds)
                fold_rows.append(FoldResult(
                    model="RFXG",
                    fold_train_end_year=fold_end_year,
                    validation_years=f"{fold_end_year+1}-{fold_end_year+2}",
                    **m,
                ))

        X_train_tab, X_val_tab, y_train_tab, y_val_tab, _ = build_tabular_arrays(
            fold_train, fold_val, categorical_cols=categorical_cols
        )

        X_tr_seq, y_tr_seq, X_vl_seq, y_vl_seq = make_sequence_arrays(
            fold_train,
            fold_val,
            group_cols=group_cols,
            categorical_cols=categorical_cols,
        )
        if len(X_tr_seq) > 0 and len(X_vl_seq) > 0:
            for name, builder in SEQUENCE_MODELS.items():
                seq_run_df, seq_mean_preds = fit_seq_runs(
                    builder, X_tr_seq, y_tr_seq, X_vl_seq, y_vl_seq,
                    model_label=name, stage_label=stage_label,
                )
                if seq_run_df.empty:
                    continue
                seq_run_df["model"] = name
                seq_run_df["fold_train_end_year"] = fold_end_year
                seq_run_df["validation_years"] = f"{fold_end_year+1}-{fold_end_year+2}"
                seq_run_df["stage"] = "cv"
                dl_run_rows.extend(seq_run_df.to_dict("records"))

                if np.all(np.isfinite(seq_mean_preds)):
                    seq_ensemble_metrics = metrics_dict(y_vl_seq, seq_mean_preds)
                    fold_rows.append(FoldResult(
                        model=name,
                        fold_train_end_year=fold_end_year,
                        validation_years=f"{fold_end_year+1}-{fold_end_year+2}",
                        **seq_ensemble_metrics,
                    ))

    cv_df = pd.DataFrame([row.__dict__ for row in fold_rows])
    dl_runs_df = pd.DataFrame(dl_run_rows)
    return cv_df, dl_runs_df


---
## 10. Final Model Fitting & Test Evaluation

In [ ]:
def fit_final_models(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    group_cols: list | None = None,
    categorical_cols: list | None = None,
    report_group_label: str = "CROP",
) -> tuple:
    """
    Train all models on the full training set and evaluate on the
    untouched final test block (2020-2024).
    """
    if group_cols is None:
        group_cols = ["DISTRICT", "CROP"]

    train_df, test_df = add_train_only_anomalies(
        train_df,
        test_df,
        anomaly_cols=CLIMATE_COLS,
        group_cols=group_cols,
    )
    stage_label = "Final Test 2020-2024"
    print(f"\n--- {stage_label} ---")

    feature_cols = build_feature_columns(train_df)
    preprocessor, _, _ = build_preprocessor(
        train_df[feature_cols], categorical_cols=categorical_cols
    )
    models = make_models(preprocessor)

    summary_rows = []
    crop_rows = []
    dl_test_run_rows = []
    report_col_exists = report_group_label in test_df.columns
    fitted_models = {}

    def _append_crop_rows(model_name, report_df):
        if report_col_exists:
            for crop, grp in report_df.groupby(report_group_label):
                m = metrics_dict(grp["YIELD"].to_numpy(), grp["pred"].to_numpy())
                crop_rows.append({"model": model_name, "crop": crop, **m})
        else:
            m = metrics_dict(report_df["YIELD"].to_numpy(), report_df["pred"].to_numpy())
            crop_rows.append({"model": model_name, "crop": "ALL", **m})

    for model_name, model in models.items():
        model.fit(train_df[feature_cols], train_df["YIELD"])
        fitted_models[model_name] = model
        preds = model.predict(test_df[feature_cols])
        overall = metrics_dict(test_df["YIELD"].to_numpy(), preds)
        summary_rows.append({"model": model_name, **overall})
        _append_crop_rows(model_name, test_df.assign(pred=preds))

    if "RF" in fitted_models and "XGB" in fitted_models:
        preds = finite_or_none((
            fitted_models["RF"].predict(test_df[feature_cols])
            + fitted_models["XGB"].predict(test_df[feature_cols])
        ) / 2.0)
        if preds is not None:
            overall = metrics_dict(test_df["YIELD"].to_numpy(), preds)
            summary_rows.append({"model": "RFXG", **overall})
            _append_crop_rows("RFXG", test_df.assign(pred=preds))

    X_train_tab, X_test_tab, y_train_tab, y_test_tab, _ = build_tabular_arrays(
        train_df, test_df, categorical_cols=categorical_cols
    )

    X_tr_seq, y_tr_seq, X_ts_seq, y_ts_seq = make_sequence_arrays(
        train_df,
        test_df,
        group_cols=group_cols,
        categorical_cols=categorical_cols,
    )
    if len(X_tr_seq) > 0 and len(X_ts_seq) > 0:
        seq_eval_df = (
            test_df
            .sort_values(group_cols + ["YEAR"])
            .reset_index(drop=True)[group_cols + ["YEAR"]]
            .copy()
        )
        for name, builder in SEQUENCE_MODELS.items():
            seq_run_df, seq_mean_preds = fit_seq_runs(
                builder, X_tr_seq, y_tr_seq, X_ts_seq, y_ts_seq,
                model_label=name, stage_label=stage_label,
            )
            if seq_run_df.empty:
                continue
            seq_run_df["model"] = name
            seq_run_df["stage"] = "test"
            dl_test_run_rows.extend(seq_run_df.to_dict("records"))

            if np.all(np.isfinite(seq_mean_preds)):
                seq_ensemble_metrics = metrics_dict(y_ts_seq, seq_mean_preds)
                summary_rows.append({"model": name, **seq_ensemble_metrics})
                merged = seq_eval_df.copy()
                merged["pred"] = seq_mean_preds
                merged["YIELD"] = y_ts_seq
                _append_crop_rows(name, merged)

    summary_df = (
        pd.DataFrame(summary_rows)
        .sort_values(["r2", "rmse"], ascending=[False, True])
        .reset_index(drop=True)
    )
    crop_df = (
        pd.DataFrame(crop_rows)
        .sort_values(["model", "crop"])
        .reset_index(drop=True)
    )
    dl_test_runs_df = pd.DataFrame(dl_test_run_rows)
    return summary_df, crop_df, dl_test_runs_df


---
## Experiment Runner & Reporting

In [ ]:
def run_experiment(
    engineered_df: pd.DataFrame,
    label: str,
    subset_filter=None,
    group_cols: list | None = None,
    categorical_cols: list | None = None,
) -> tuple:
    """
    End-to-end experiment wrapper.
    """
    exp_df = engineered_df.loc[subset_filter].copy() if subset_filter is not None else engineered_df.copy()

    if categorical_cols is not None and "CROP" not in categorical_cols and "CROP" in exp_df.columns:
        exp_df = exp_df.drop(columns=["CROP"])

    train_df = exp_df[exp_df["YEAR"] < FINAL_TEST_START].copy()
    test_df = exp_df[exp_df["YEAR"] >= FINAL_TEST_START].copy()

    print(f"\n{'='*60}")
    print(f"  Experiment : {label.upper()}")
    print(f"  Train rows : {len(train_df):,} | Test rows: {len(test_df):,}")
    print(
        f"  Train years: {train_df['YEAR'].min()}-{train_df['YEAR'].max()} "
        f"| Test years: {test_df['YEAR'].min()}-{test_df['YEAR'].max()}"
    )
    print(f"{'='*60}")

    cv_df, cv_dl_runs_df = run_cross_validation(
        train_df,
        build_feature_columns(train_df),
        group_cols=group_cols,
        categorical_cols=categorical_cols,
    )
    test_summary_df, crop_df, test_dl_runs_df = fit_final_models(
        train_df,
        test_df,
        group_cols=group_cols,
        categorical_cols=categorical_cols,
    )

    slug = slugify(label)
    cv_df.to_csv(OUTPUT_DIR / f"walk_forward_results_{slug}.csv", index=False)
    test_summary_df.to_csv(OUTPUT_DIR / f"final_test_results_{slug}.csv", index=False)
    crop_df.to_csv(OUTPUT_DIR / f"per_crop_test_results_{slug}.csv", index=False)
    cv_dl_runs_df.to_csv(OUTPUT_DIR / f"dl_cv_run_metrics_{slug}.csv", index=False)
    test_dl_runs_df.to_csv(OUTPUT_DIR / f"dl_test_run_metrics_{slug}.csv", index=False)

    return cv_df, test_summary_df, cv_dl_runs_df, test_dl_runs_df


In [ ]:
def write_report(
    profile: dict,
    engineered_df: pd.DataFrame,
    cv_df: pd.DataFrame,
    test_summary_df: pd.DataFrame,
    crop_df: pd.DataFrame,
) -> None:
    """Write a Markdown summary report to OUTPUT_DIR."""
    report_path = OUTPUT_DIR / "crop_yield_rebuild_report.md"
    best_model = test_summary_df.iloc[0]["model"]

    cv_mean = (
        cv_df.groupby("model")[["rmse", "mae", "r2"]]
        .mean()
        .sort_values(["r2", "rmse"], ascending=[False, True])
        .round(4)
    )

    def block_table(df, index=True):
        return "```\n" + df.to_string(index=index) + "\n```"

    base_r2 = PERSISTENCE_BASELINE["r2"]
    base_rmse = PERSISTENCE_BASELINE["rmse"]

    lines = [
        "# Leakage-Safe Crop Yield Prediction Report",
        "",
        "## Dataset Audit",
        f"- Raw shape: {tuple(profile['shape'])}",
        f"- Year range: {profile['year_min']} to {profile['year_max']}",
        f"- Districts: {profile['districts']}",
        f"- Crops: {profile['crops']}",
        f"- District-crop panels: {profile['district_crop_panels']}",
        f"- Constant numeric columns dropped: {profile['constant_numeric_columns']}",
        "- Multicollinear columns dropped: P_MINUS_PET, TMX, TMN, VAP_KPA",
        f"- Mean |PRODUCTION/AREA - YIELD|: {profile['yield_formula_mean_abs_error']:.4f}",
        f"- Rows with AREA < 1: {profile['rows_with_area_below_1']}",
        "",
        "## EDA Summary",
        f"- Retained climate features after multicollinearity reduction: {CLIMATE_COLS}",
        "- All feature-selection diagnostics were restricted to pre-2020 training years.",
        "- Outlier structure documented; no rows removed (outliers are real agronomic events).",
        "- Multi-method correlation confirmed both linear and non-linear climate signals.",
        "",
        "## Persistence Baseline (YIELD_lag1 on final test block)",
        f"- R2={base_r2:.4f}  RMSE={base_rmse:.4f}",
        "- Any model must exceed this benchmark on the same 2020-2024 holdout block.",
        "",
        "## Method Choices",
        "- Strict temporal holdout: train 2003-2019, test 2020-2024 (after lag generation).",
        "- All lag/rolling features use shift(1); current row never sees its own target.",
        "- Anomaly features computed from training group means only; frozen before test application.",
        "- ANN and all sequence DL models averaged over 10 independent seeds.",
        "- Sequence scaling is fit on training rows only and sequence windows include the prediction-year features.",
        "- Walk-forward CV with 3 folds inside training years only.",
        "",
        "## Engineered Dataset",
        f"- Modeling rows: {len(engineered_df):,}",
        f"- Modeling columns: {engineered_df.shape[1]}",
        "",
        "## Walk-Forward CV Mean Metrics",
        block_table(cv_mean),
        "",
        "## Final Test Block (2020-2024)",
        block_table(test_summary_df.round(4), index=False),
        "",
        "## Per-Crop Test Metrics",
        block_table(crop_df.round(4), index=False),
        "",
        "## Verdict",
        f"- Best model on untouched test block: `{best_model}`.",
        f"- Persistence baseline R2 on the same holdout block: {base_r2:.4f}.",
        "- For annual panel data, leakage-safe tabular models often remain the strongest benchmark.",
    ]
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Report saved -> {report_path}")


---
## Run Experiment

In [ ]:
cv_df_pooled, test_summary_pooled, cv_dl_runs_pooled, test_dl_runs_pooled = run_experiment(
    engineered_df,
    label="pooled",
    group_cols=["DISTRICT", "CROP"],
    categorical_cols=["DISTRICT", "CROP"],
)



  Experiment : POOLED
  Train rows : 1,037 | Test rows: 305
  Train years: 2003-2019 | Test years: 2020-2024

--- CV 2014-2015 ---
[CV 2014-2015] LSTM run 1/10 started
[CV 2014-2015] LSTM run 1/10 finished: epochs=21  RMSE=4.2052  MAE=2.2116  R2=0.9621
[CV 2014-2015] LSTM run 2/10 started
[CV 2014-2015] LSTM run 2/10 finished: epochs=36  RMSE=4.4358  MAE=2.4968  R2=0.9578
[CV 2014-2015] LSTM run 3/10 started
[CV 2014-2015] LSTM run 3/10 finished: epochs=30  RMSE=5.0911  MAE=2.9950  R2=0.9444
[CV 2014-2015] LSTM run 4/10 started
[CV 2014-2015] LSTM run 4/10 finished: epochs=19  RMSE=4.2768  MAE=2.6193  R2=0.9608
[CV 2014-2015] LSTM run 5/10 started
[CV 2014-2015] LSTM run 5/10 finished: epochs=21  RMSE=4.1824  MAE=2.4153  R2=0.9625
[CV 2014-2015] LSTM run 6/10 started
[CV 2014-2015] LSTM run 6/10 finished: epochs=26  RMSE=4.4797  MAE=2.4999  R2=0.9570
[CV 2014-2015] LSTM run 7/10 started
[CV 2014-2015] LSTM run 7/10 finished: epochs=41  RMSE=4.9374  MAE=2.9702  R2=0.9477
[CV 2014-2015]

In [ ]:
print("\nWalk-forward CV (pooled) - mean across folds")
cv_mean_pooled = (
    cv_df_pooled.groupby("model")[["rmse", "mae", "r2"]]
    .mean()
    .sort_values(["r2", "rmse"], ascending=[False, True])
    .round(4)
)
display(cv_mean_pooled)

print("\nDL walk-forward run metrics (pooled)")
display(
    cv_dl_runs_pooled
    .sort_values(["model", "fold_train_end_year", "run"])
    .reset_index(drop=True)
)

print("\nDL walk-forward average metrics (pooled)")
display(
    cv_dl_runs_pooled
    .groupby(["model", "fold_train_end_year", "validation_years"])[["rmse", "mae", "r2"]]
    .mean()
    .round(4)
    .reset_index()
)



Walk-forward CV (pooled) - mean across folds


,rmse,mae,r2
model,,,
GB,4.9078,2.0528,0.9531
RCNN,4.9855,2.3694,0.9501
GRU,5.0479,2.7458,0.9493
LSTM,5.0770,2.5369,0.9488
CNN-BiLSTM,5.0729,2.4787,0.9484
1DCNN,5.2297,2.6843,0.9464
XGB,5.4030,2.2379,0.9438
CNN-Transformer,5.2951,2.5768,0.9431
RFXG,5.4536,2.1862,0.9415



DL walk-forward run metrics (pooled)


,run,epochs_used,rmse,mae,r2,model,fold_train_end_year,validation_years,stage
0,1,52,4.942420,2.970004,0.947644,1DCNN,2013,2014-2015,cv
1,2,45,4.874256,2.794661,0.949079,1DCNN,2013,2014-2015,cv
2,3,26,4.505204,2.828952,0.956498,1DCNN,2013,2014-2015,cv
3,4,59,5.569122,3.119045,0.933525,1DCNN,2013,2014-2015,cv
4,5,26,8.311285,5.636348,0.851946,1DCNN,2013,2014-2015,cv
...,...,...,...,...,...,...,...,...,...
175,6,21,3.653428,2.358300,0.973899,RCNN,2017,2018-2019,cv
176,7,40,3.051932,1.828429,0.981786,RCNN,2017,2018-2019,cv
177,8,30,3.631720,2.388138,0.974208,RCNN,2017,2018-2019,cv
178,9,28,3.602141,2.211922,0.974627,RCNN,2017,2018-2019,cv



DL walk-forward average metrics (pooled)


,model,fold_train_end_year,validation_years,rmse,mae,r2
0,1DCNN,2013,2014-2015,5.5086,3.5242,0.9324
1,1DCNN,2015,2016-2017,8.0931,4.0784,0.8971
2,1DCNN,2017,2018-2019,4.2436,2.9927,0.9642
3,CNN-BiLSTM,2013,2014-2015,4.7189,2.7598,0.9512
4,CNN-BiLSTM,2015,2016-2017,8.0503,3.6944,0.8980
5,CNN-BiLSTM,2017,2018-2019,3.5460,2.2599,0.9752
6,CNN-Transformer,2013,2014-2015,5.3826,3.1194,0.9364
7,CNN-Transformer,2015,2016-2017,8.1823,3.7687,0.8948
8,CNN-Transformer,2017,2018-2019,3.5088,2.2879,0.9755
9,GRU,2013,2014-2015,4.6640,3.0199,0.9530


In [ ]:
print("\nFinal test results (pooled) - 2020-2024")
display(test_summary_pooled.round(4))

print("\nDL final-test run metrics (pooled)")
display(
    test_dl_runs_pooled
    .sort_values(["model", "run"])
    .reset_index(drop=True)
)

print("\nDL final-test average metrics (pooled)")
display(
    test_dl_runs_pooled
    .groupby("model")[["rmse", "mae", "r2"]]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(["r2", "rmse"], ascending=[False, True])
    .reset_index(drop=True)
)



Final test results (pooled) - 2020-2024


,model,rmse,mae,r2
0,1DCNN,15.1535,5.2848,0.7331
1,CNN-BiLSTM,15.3145,5.3184,0.7274
2,CNN-Transformer,15.3146,5.3346,0.7274
3,LSTM,15.3386,5.3484,0.7265
4,RCNN,15.3709,5.1843,0.7254
5,RF,15.4209,4.4173,0.7236
6,GRU,15.4294,5.7227,0.7233
7,RFXG,15.5717,4.5622,0.7182
8,XGB,15.8561,4.8898,0.7078
9,GB,15.8674,4.7068,0.7073



DL final-test run metrics (pooled)


,run,epochs_used,rmse,mae,r2,model,stage
0,1,28,16.107400,6.103562,0.698425,1DCNN,test
1,2,29,15.725136,6.464840,0.712570,1DCNN,test
2,3,44,15.713562,5.740451,0.712993,1DCNN,test
3,4,35,15.766354,5.750534,0.711061,1DCNN,test
4,5,55,15.345659,5.859674,0.726275,1DCNN,test
5,6,43,15.402866,6.021964,0.724230,1DCNN,test
6,7,29,16.143297,6.513011,0.697080,1DCNN,test
7,8,35,15.433414,5.889503,0.723135,1DCNN,test
8,9,20,15.591796,6.953361,0.717423,1DCNN,test
9,10,37,15.147961,5.632873,0.733282,1DCNN,test



DL final-test average metrics (pooled)


,model,rmse,mae,r2
0,CNN-Transformer,15.4290,5.4715,0.7233
1,LSTM,15.4652,5.4661,0.7219
2,CNN-BiLSTM,15.4766,5.5580,0.7214
3,RCNN,15.4913,5.3541,0.7210
4,GRU,15.5433,5.8285,0.7191
5,1DCNN,15.6377,6.0930,0.7156
